ASX200 Similarity Engine & Snapshot Analysis 

This notebook starts with the Live Snapshot & SIIF portfolio validation. 

The next component is then the Similarity Engine, which consists of: 
1. Historical Z-score database - computes rolling metrics for all 189 tickers 
2. KNN index - fit nearest neighbour model in Z-score space
3. Sanity Check - validates the analogues returned 


It uses a live market-state 'snapshot' input to query our historical database and find the most similar historical analogues using Euclidean distance. 


In [9]:
import yfinance as yf 
import pandas as pd
import numpy as np 
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [10]:
#METRIC CALCULATIONS
def get_current_snapshot(ticker: str) -> dict:
    yt = ticker if ticker.endswith(".AX") else ticker + ".AX"
    market_ticker = "^AXJO"
    
    # Download 2y to have enough history for the Z-score means
    raw = yf.download([yt, market_ticker], period="2y", auto_adjust=True, progress=False)
    
    if raw.empty or yt not in raw['Close']:
        raise ValueError(f"Ticker {yt} data missing.")

    prices = raw["Close"]
    volumes = raw["Volume"]
    
    # Daily Returns
    stock_ret = prices[yt].pct_change().dropna()
    market_ret = prices[market_ticker].pct_change().dropna()

    # 1. Momentum Z-Score
    rolling_12m = stock_ret.rolling(252).apply(lambda x: (1 + x).prod() - 1).dropna()
    momentum_zscore = (rolling_12m.iloc[-1] - rolling_12m.mean()) / rolling_12m.std()

    # 2. Volatility Z-Score
    rolling_vol = stock_ret.rolling(30).std() * np.sqrt(252)
    rolling_vol = rolling_vol.dropna()
    vol_zscore = (rolling_vol.iloc[-1] - rolling_vol.mean()) / rolling_vol.std()

    # 3. Idiosyncratic Volatility (Residual from Market Regression)
    aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()
    aligned.columns = ['stock', 'market']
    X, y = aligned['market'], aligned['stock']
    
    cov_matrix = np.cov(y, X)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1]
    
    alpha = y.mean() - beta * X.mean()
    residuals = y - (alpha + beta * X)
    idio_vol = residuals.std() * np.sqrt(252)

    # 4. Liquidity Proxy (Average Daily Dollar Volume)
    dollar_volume = (prices[yt] * volumes[yt]).rolling(30).mean().iloc[-1]

    return {
        "ticker": ticker,
        "date": str(prices.index[-1].date()),
        "momentum_z": round(float(momentum_zscore), 4),
        "volatility_z": round(float(vol_zscore), 4),
        "idio_vol": round(float(idio_vol), 4),
        "dollar_vol_30d": round(float(dollar_volume), 2)
    }


Portfolio Validation

The following code shows the validation for SIIF's current portfolio (comprised of 10 stocks, as opposed to all ASX200) 

In [ ]:
# SIIF CURRENT PORTFOLIO HOLDINGS
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]
results = []

print("--- Extracting Live Metrics for SIIF Portfolio ---")
for t in portfolio_tickers:
    try:
        data = get_current_snapshot(t)
        results.append(data)
        print(f"✅ {t:4} | Snapshot captured.")
    except Exception as e:
        print(f"❌ {t:4} | Error: {e}")

# Results converted to table 
portfolio_df = pd.DataFrame(results)
display(portfolio_df)

portfolio_df.to_csv("portfolio_snapshot.csv", index=False)


--- Extracting Live Metrics for SMIF Portfolio ---
✅ PLY  | Snapshot captured.
✅ LAU  | Snapshot captured.
✅ COS  | Snapshot captured.
✅ ANG  | Snapshot captured.
✅ VVA  | Snapshot captured.
✅ WTC  | Snapshot captured.
✅ AUB  | Snapshot captured.
✅ TLX  | Snapshot captured.


/var/folders/65/_xrrhj392r9ggvzk0zdwjjph0000gn/T/ipykernel_22984/2801229388.py:29: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()


✅ XYZ  | Snapshot captured.
✅ DUG  | Snapshot captured.


,ticker,date,momentum_z,volatility_z,idio_vol,dollar_vol_30d
0,PLY,2026-05-08,2.2553,1.3304,0.8906,166437.56
1,LAU,2026-05-08,0.9667,-0.1391,0.3633,287416.92
2,COS,2026-05-08,-1.4644,0.5512,0.6314,19523.97
3,ANG,2026-05-08,-0.3517,-0.5495,0.6005,168877.26
4,VVA,2026-05-08,0.4527,-1.0055,0.4109,199883.76
5,WTC,2026-05-08,-1.0743,1.3747,0.4832,66337903.56
6,AUB,2026-05-08,-2.2815,-0.2926,0.2904,10488458.70
7,TLX,2026-05-08,-0.4840,0.0369,0.5051,36887232.00
8,XYZ,2026-05-08,1.6368,-0.6906,0.5743,13896714.16
9,DUG,2026-05-08,1.3512,-0.3894,0.5551,322747.12


Similarity Engine 

Building the KNN-based historical analogue matching system.

1. Historical Z-score Database

For each ticker in the master dataset, four rolling metrics are computed at each historical date 
- Momentum (12-month returns relative to historical mean)
- Volatility (30-day realised volatility relative to historical mean)
- Idiosyncratic Volatility (Residual Alpha after removing market influence) 
- Liquidity Proxy (30-day average daily dollar volume) 

All Z-scores use an expanding window (as opposed to rolling) to avoid lookahead bias. Each point is only normalised against its own past.

In [27]:
#HISTORICAL Z-SCORE DATABASE 
def compute_historical_zscores(master_path: str = "ASX200_MASTER_DATASET.csv",
                                market_ticker: str = "^AXJO") -> pd.DataFrame:
    print("Loading master dataset...")
    master = pd.read_csv(master_path, parse_dates=["Date"])
    master = master.sort_values(["ticker", "Date"]).reset_index(drop=True)

    print("Downloading market index...")
    mkt_raw = yf.download(market_ticker, period="5y", auto_adjust=True, progress=False)
    if isinstance(mkt_raw.columns, pd.MultiIndex):
        mkt_raw.columns = mkt_raw.columns.get_level_values(0)
    market_ret = mkt_raw["Close"].pct_change().dropna()
    market_ret.index = market_ret.index.tz_localize(None)

    all_records = []
    tickers = master["ticker"].unique()
    print(f"Computing Z-scores for {len(tickers)} tickers...")

    for ticker in tickers:
        try:
            df = master[master["ticker"] == ticker].copy()
            df = df.set_index("Date").sort_index()

            if len(df) < 300:
                continue

            stock_ret = df["Close"].pct_change()
            aligned = pd.concat([stock_ret, market_ret], axis=1, sort=True).dropna()
            aligned.columns = ["stock", "market"]

            rolling_12m = stock_ret.rolling(252).apply(lambda x: (1 + x).prod() - 1)
            rolling_vol = stock_ret.rolling(30).std() * np.sqrt(252)

            def rolling_idio(idx):
                window = aligned.loc[:idx].tail(252)
                if len(window) < 60:
                    return np.nan
                s, m = window["stock"], window["market"]
                beta = np.cov(s, m)[0, 1] / np.var(m)
                alpha = s.mean() - beta * m.mean()
                residuals = s - (alpha + beta * m)
                return residuals.std() * np.sqrt(252)

            idio_series = pd.Series(
                [rolling_idio(idx) for idx in df.index],
                index=df.index
            )

            dollar_vol = (df["Close"] * df["Volume"]).rolling(30).mean()

            def expanding_zscore(s):
                return (s - s.expanding().mean()) / s.expanding().std()

            records = pd.DataFrame({
                "ticker": ticker,
                "momentum_z": expanding_zscore(rolling_12m),
                "volatility_z": expanding_zscore(rolling_vol),
                "idio_vol_z": expanding_zscore(idio_series),
                "dollar_vol_z": expanding_zscore(np.log1p(dollar_vol)),
            }, index=df.index)

            records = records.dropna()
            all_records.append(records)

        except Exception as e:
            print(f"❌ {ticker}: {e}")

    db = pd.concat(all_records).reset_index().rename(columns={"index": "Date"})
    print(f"\n✅ Z-score database built: {db['ticker'].nunique()} tickers, {len(db):,} snapshots")
    db.to_csv("historical_zscores.csv", index=False)
    return db

The first run of the next code will be slow for all 189 tickers. The output is saved to 'historical_zscores.parquet'. 

If you intend to run this notebook multiple times, skip this cell and load directly from the csv.

In [21]:
historical_db = compute_historical_zscores("ASX200_MASTER_DATASET.csv")

Loading master dataset...
Computing Z-scores for 189 tickers...


/var/folders/65/_xrrhj392r9ggvzk0zdwjjph0000gn/T/ipykernel_22984/1844723757.py:28: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()
/var/folders/65/_xrrhj392r9ggvzk0zdwjjph0000gn/T/ipykernel_22984/1844723757.py:28: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()
/var/folders/65/_xrrhj392r9ggvzk0zdwjjph0000gn/T/ipykernel_22984/1844723757.py:28: Pandas4Warning: Sorting by defa


✅ Z-score database built: 189 tickers, 95,734 snapshots


2. KNN Index

We are fitting a K-Nearest Neighbours model on the full historical Z-score database using Euclidean distance in Z-score space.

Given a live snapshot of any ASX stock today, the search function returns the K most similar historical observations across all tickers and dates.

Note: self-matches (same ticker) are excluded from returned analogues.

In [23]:
#KNN INDEX BUILDING AND ANALOGUE SEARCH
FEATURE_COLS = ["momentum_z", "volatility_z", "idio_vol_z", "dollar_vol_z"]

def build_knn_index(historical_db: pd.DataFrame, k: int = 10):
    db_clean = historical_db[FEATURE_COLS + ["ticker", "Date"]].dropna().copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(db_clean[FEATURE_COLS])
    knn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    knn.fit(X)
    print(f"✅ KNN index built on {len(db_clean):,} historical snapshots")
    return knn, scaler, db_clean


def find_analogues(snapshot: dict, knn, scaler, db_clean: pd.DataFrame, k: int = 10) -> pd.DataFrame:
    query = np.array([[
        snapshot["momentum_z"],
        snapshot["volatility_z"],
        snapshot.get("idio_vol_z", snapshot["idio_vol"]),
        snapshot.get("dollar_vol_z", 0)
    ]])

    query_scaled = scaler.transform(query)
    distances, indices = knn.kneighbors(query_scaled, n_neighbors=min(k + 20, len(db_clean)))

    analogues = db_clean.iloc[indices[0]].copy()
    analogues["distance"] = distances[0]
    analogues = analogues[analogues["ticker"] != snapshot["ticker"]]
    analogues = analogues.head(k).reset_index(drop=True)
    return analogues

In [24]:
#Fitting the KNN index on the historical Z-score databse
knn, scaler, db_clean = build_knn_index(historical_db, k=10)

✅ KNN index built on 95,734 historical snapshots


3. Sanity Check 

This code runs the similarity engine on a handful of stocks and manually inspects whether the analogues make intuitive sense.

In [25]:
# SANITY CHECK: FIND ANALOGUES FOR A PORTFOLIO SNAPSHOT
def run_sanity_check(snapshot: dict, knn, scaler, db_clean: pd.DataFrame, k: int = 10):
    print(f"\n{'='*60}")
    print(f"  ANALOGUES FOR: {snapshot['ticker']}  (as of {snapshot['date']})")
    print(f"{'='*60}")
    print(f"  momentum_z  : {snapshot['momentum_z']}")
    print(f"  volatility_z: {snapshot['volatility_z']}")
    print(f"  idio_vol    : {snapshot['idio_vol']}")
    print(f"  dollar_vol  : {snapshot['dollar_vol_30d']:,.0f}")
    print(f"{'-'*60}")

    analogues = find_analogues(snapshot, knn, scaler, db_clean, k)
    print(analogues[["ticker", "Date", "momentum_z", "volatility_z", "distance"]].to_string(index=False))
    return analogues

In [26]:
#USING SIIF PORTFOLIO SNAPSHOT FOR SANITY CHECK
portfolio_tickers = ["PLY", "LAU", "COS", "ANG", "VVA", "WTC", "AUB", "TLX", "XYZ", "DUG"]

for t in portfolio_tickers:
    try:
        snap = get_current_snapshot(t)
        analogues = find_analogues(snap, knn, scaler, db_clean, k=10)
        print(f"\n{'='*60}")
        print(f"  ANALOGUES FOR: {t}  (as of {snap['date']})")
        print(f"  momentum_z: {snap['momentum_z']}  |  volatility_z: {snap['volatility_z']}  |  idio_vol: {snap['idio_vol']}")
        print(f"{'='*60}")
        display(analogues[["ticker", "Date", "momentum_z", "volatility_z", "idio_vol_z", "distance"]])
    except Exception as e:
        print(f"❌ {t}: {e}")


  ANALOGUES FOR: PLY  (as of 2026-05-08)
  momentum_z: 2.2553  |  volatility_z: 1.3304  |  idio_vol: 0.8906


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,ticker,Date,momentum_z,volatility_z,idio_vol_z,distance
0,DNL,2026-03-06,2.320983,1.359379,0.744138,0.122349
1,DNL,2026-03-05,2.400242,1.371170,0.774767,0.163260
2,DNL,2026-03-04,2.063533,1.451373,0.757013,0.211741
3,DNL,2026-03-02,2.207520,1.157461,0.699452,0.259572
4,APA,2025-06-11,2.140376,1.468400,1.132483,0.274866
5,NHF,2025-10-10,2.151085,1.619361,0.853531,0.275546
6,DNL,2026-03-03,2.564403,1.133666,0.716615,0.338907
7,WOW,2025-07-07,2.228811,0.962067,1.104977,0.347220
8,WOW,2026-01-30,1.902514,1.363104,0.667976,0.350433
9,STO,2024-07-15,1.918477,1.202465,0.613359,0.367307



  ANALOGUES FOR: LAU  (as of 2026-05-08)
  momentum_z: 0.9667  |  volatility_z: -0.1391  |  idio_vol: 0.3633


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


...

/var/folders/65/_xrrhj392r9ggvzk0zdwjjph0000gn/T/ipykernel_22984/2801229388.py:29: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  aligned = pd.concat([stock_ret, market_ret], axis=1).dropna()
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,ticker,Date,momentum_z,volatility_z,idio_vol_z,distance
0,DRR,2025-07-25,1.608599,-0.550906,0.473755,0.170778
1,GPT,2025-11-27,1.514296,-0.683907,0.751894,0.216923
2,GPT,2025-11-28,1.791024,-0.687854,0.750872,0.245129
3,GPT,2026-05-01,1.363154,-0.899389,0.732673,0.299121
4,SCG,2025-11-27,1.435915,-0.770081,0.315224,0.304666
5,GPT,2025-11-26,1.291248,-0.686290,0.753003,0.313632
6,VCX,2025-08-29,1.588208,-0.500959,0.625428,0.323644
7,ALQ,2024-11-15,1.559187,-0.811639,0.524256,0.331462
8,MIN,2026-06-05,1.650018,-0.996679,0.441487,0.332273
9,BHP,2025-08-21,1.551870,-0.399242,0.606133,0.338491



  ANALOGUES FOR: DUG  (as of 2026-05-08)
  momentum_z: 1.3512  |  volatility_z: -0.3894  |  idio_vol: 0.5551


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,ticker,Date,momentum_z,volatility_z,idio_vol_z,distance
0,CIA,2026-05-05,1.316044,-0.279990,0.343644,0.194630
1,AIA,2025-09-05,1.246830,-0.140419,0.525241,0.224878
2,BHP,2025-08-20,1.511514,-0.186740,0.628331,0.228136
3,BHP,2025-06-10,1.529491,-0.210643,0.704275,0.233451
4,AIA,2025-07-21,1.116381,-0.473454,0.705368,0.237335
5,BHP,2025-06-11,1.442592,-0.194968,0.693316,0.245827
6,CIP,2024-09-02,1.435585,-0.437174,0.824078,0.259735
7,DRR,2025-07-25,1.608599,-0.550906,0.473755,0.265984
8,BHP,2025-06-06,1.103085,-0.453075,0.655155,0.272180
9,BHP,2025-08-21,1.551870,-0.399242,0.606133,0.275789


10/05/2026 Sanity Check Note:

Results look sensible across the portfolio: 
- **PLY** (momentum_z: 2.26, high vol): analogues are DNL consecutive days + APA, WOW — stocks in strong upward momentum phases
- **COS** (momentum_z: -1.46, mild vol): analogues are S32, GMG, WES, MIN — large caps in drawdown periods 
- **AUB** (momentum_z: -2.28, low vol): analogues are CHC, BWP, AFI — defensive/financial names in downtrends 
- **DUG** (momentum_z: 1.35, low vol): BHP appearing 5 times — DUG being matched to BHP in mid-2025 momentum phase 

No obviously wrong matches (e.g. mining stocks appearing as analogues for financials). 
Similarity engine returning coherent results. 

Proceeding to Week 3: forward return distributions.

Known limitations at this stage:
- dollar_vol_z not yet included in KNN query 
